# Dependencies

**Objective:** Declare and isolate every runtime dependency so a clean environment can reproduce the application.

## Simple version

Separate packages required by the running service from tools used only during development.

In [ ]:
# Runtime packages run the service; development packages support local work.
dependencies = {
    "runtime": ["fastapi", "httpx"],
    "development": ["pytest", "ruff"],
}

print(dependencies)

## Polished version

The manifest is the dependency contract; the lockfile pins the exact resolved versions used to build a release.

In [ ]:
import tomllib
from dataclasses import dataclass


# This is the same structure used by a real pyproject.toml file.
manifest_text = """
[project]
name = "example-api"
requires-python = ">=3.12"
dependencies = [
    "fastapi>=0.116,<1",
    "httpx>=0.28,<1",
]

[dependency-groups]
dev = [
    "pytest>=8.4,<9",
    "ruff>=0.12,<1",
]
"""


# Normalize the parsed manifest into a small typed object.
@dataclass(frozen=True)
class DependencyManifest:
    runtime: tuple[str, ...]
    development: tuple[str, ...]
    python: str

    @classmethod
    def parse(cls, text: str) -> "DependencyManifest":
        data = tomllib.loads(text)
        project = data["project"]
        return cls(
            runtime=tuple(project["dependencies"]),
            development=tuple(data["dependency-groups"]["dev"]),
            python=project["requires-python"],
        )


manifest = DependencyManifest.parse(manifest_text)
# --frozen enforces the lockfile; uv run uses the isolated environment.
commands = ["uv sync --frozen", "uv run uvicorn app.main:app"]

print(manifest)
print("Isolated commands:", commands)

## Applied in this repository

The root and both projects declare dependencies in `pyproject.toml`, pin them in `uv.lock`, and run commands through `uv run` instead of relying on globally installed packages.